# 06. Urine Output Raw (소변량 원본 추출)

## 목적
코호트 환자의 소변량 데이터를 **1시간 단위**로 추출

## 입력
- `cohort_base.csv`: 기본 코호트

## 출력
- `urine_raw.csv`: 시간별 소변량 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (outputevents)
- [x] 시간당 합산
- [x] 체중 데이터 병합 (mL/kg/hr 계산용)
- [ ] 슬라이딩 윈도우 적용 → merge 단계에서

## Item IDs
| Item ID | 설명 |
|---------|------|
| 226559 | Urine Out (Foley) - 소변줄 |
| 226560 | Urine Out (Void) - 자연 배뇨 |

In [ ]:
import duckdb
import pandas as pd
import os

DB_PATH = '../data/duckdb/mimic_total.duckdb'
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

con = duckdb.connect(DB_PATH)
print("=== 06. Urine Output Raw 추출 시작 ===")

## Step 1: 코호트 로드

In [ ]:
print("Step 1: 코호트 로드")

cohort_path = os.path.join(INPUT_DIR, 'cohort_base.csv')
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.register('cohort', df_cohort)

## Step 2: 체중 데이터 추출

In [ ]:
print("\nStep 2: 체중 데이터 추출")

weight_query = """
SELECT 
    c.stay_id,
    AVG(TRY_CAST(ce.valuenum AS DOUBLE)) as weight_kg
FROM cohort c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE ce.itemid = '226512'  -- Admission Weight
  AND TRY_CAST(ce.valuenum AS DOUBLE) > 0
  AND TRY_CAST(ce.valuenum AS DOUBLE) < 500  -- 이상치 제외
GROUP BY c.stay_id
"""

df_weight = con.execute(weight_query).df()
print(f"✓ 체중 데이터: {len(df_weight):,}명")
print(f"  - 평균 체중: {df_weight['weight_kg'].mean():.1f} kg")

## Step 3: Urine Output 추출

In [ ]:
print("\nStep 3: Urine Output 추출 (1시간 단위)")

urine_query = """
SELECT
    c.stay_id,
    date_trunc('hour', CAST(oe.charttime AS TIMESTAMP)) as charttime_h,
    SUM(TRY_CAST(oe.value AS DOUBLE)) as urine_ml
FROM cohort c
INNER JOIN outputevents oe ON c.stay_id = oe.stay_id
WHERE oe.itemid IN ('226559', '226560')  -- Foley, Void
  AND TRY_CAST(oe.value AS DOUBLE) > 0
  AND TRY_CAST(oe.value AS DOUBLE) < 5000  -- 이상치 제외 (시간당 5L 초과)
  -- ICU 체류 기간 내
  AND CAST(oe.charttime AS TIMESTAMP) >= c.intime
  AND CAST(oe.charttime AS TIMESTAMP) <= c.outtime
GROUP BY c.stay_id, charttime_h
ORDER BY c.stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_urine = con.execute(urine_query).df()

print(f"✓ Urine Output 추출 완료")
print(f"  - 총 행 수: {len(df_urine):,}개")
print(f"  - 고유 환자: {df_urine['stay_id'].nunique():,}명")

## Step 4: 체중 병합 및 mL/kg/hr 계산

In [ ]:
print("\nStep 4: 체중 병합 및 mL/kg/hr 계산")

# 체중 병합
df_urine = df_urine.merge(df_weight, on='stay_id', how='left')

# 체중 없는 경우 전체 평균으로 대체
median_weight = df_weight['weight_kg'].median()
df_urine['weight_kg'] = df_urine['weight_kg'].fillna(median_weight)

# mL/kg/hr 계산
df_urine['urine_ml_kg_hr'] = df_urine['urine_ml'] / df_urine['weight_kg']

# 올리구리아 플래그 (0.5 mL/kg/hr 미만)
df_urine['oliguria_flag'] = (df_urine['urine_ml_kg_hr'] < 0.5).astype(int)

print(f"✓ 계산 완료")
print(f"  - 평균 urine: {df_urine['urine_ml'].mean():.1f} mL/hr")
print(f"  - 평균 urine/kg: {df_urine['urine_ml_kg_hr'].mean():.2f} mL/kg/hr")
print(f"  - 올리구리아 비율: {df_urine['oliguria_flag'].mean()*100:.1f}%")

## Step 5: 저장

In [ ]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

# 컬럼 정리
df_urine = df_urine[['stay_id', 'charttime_h', 'urine_ml', 'weight_kg', 'urine_ml_kg_hr', 'oliguria_flag']]

output_path = os.path.join(OUTPUT_DIR, 'urine_raw.csv')
df_urine.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: urine_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_urine):,}개")
print(f"  - 경로: {output_path}")

In [ ]:
print("\n=== 샘플 데이터 ===")
df_urine.head(10)

In [ ]:
con.close()
print("\n=== 06. Urine Output Raw 추출 완료 ===")